In [86]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import time

from pathlib import Path
from collections import Counter
from itertools import chain, product
from tqdm.auto import tqdm

ROOT = Path.cwd().parent.resolve()
stopwords_path = ROOT / "data" / "nltk" / "eng_stopwords.txt"
runs_dir = ROOT / "hw2" / "runs"

def queries_path(folder_name):
    return ROOT / "data" / "wikIR1k" / folder_name / "queries.csv"

def qrels_path(folder_name):
    return ROOT / "data" / "wikIR1k" / folder_name / "qrels"

docs_path = ROOT / "data" / "wikIR1k" / "documents.csv"
def custom_path_documents(method):
    return ROOT / "data" / "wikIR1k_custom" / f"{method}_documents.csv"

with open(stopwords_path, "r", encoding="utf-8") as f:
    stopwords = {line.strip() for line in f if line.strip()}

In [64]:
import spacy
from nltk.stem import PorterStemmer
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
import ir_measures
from ir_measures import AP, P, nDCG

---
# Basic stats for test queries

In [30]:
def print_basic_query_stats(
    query_df: pd.DataFrame,
    qrels: pd.DataFrame,
    query_id_col: str = "id_left",
    query_text_col: str = "text_left",
    qrels_query_id_col: str = "query_id",
    qrels_doc_id_col: str = "doc_id",
    qrels_relevance_col: str = "relevance",
    digits: int = 2,
) -> tuple[dict, pd.DataFrame]:
    query_required_cols = [query_id_col, query_text_col]
    qrels_required_cols = [qrels_query_id_col, qrels_doc_id_col, qrels_relevance_col]

    missing_query_cols = [col for col in query_required_cols if col not in query_df.columns]
    missing_qrels_cols = [col for col in qrels_required_cols if col not in qrels.columns]

    assert not missing_query_cols, f"Missing columns in query_df: {missing_query_cols}"
    assert not missing_qrels_cols, f"Missing columns in qrels: {missing_qrels_cols}"
    assert not query_df[query_required_cols].isna().any().any(), f"Missing values found in {query_required_cols}"
    assert not qrels[qrels_required_cols].isna().any().any(), f"Missing values found in {qrels_required_cols}"

    query_mod_df = query_df.copy()

    tokenized = query_mod_df[query_text_col].str.split()
    query_mod_df["query_length_words"] = tokenized.apply(len)

    relevant_qrels = qrels[qrels[qrels_relevance_col] != 0]
    rel_docs_per_query = (
        relevant_qrels.groupby(qrels_query_id_col)[qrels_doc_id_col]
        .nunique()
        .rename("relevant_docs_count")
    )

    query_mod_df = query_mod_df.merge(
        rel_docs_per_query,
        how="left",
        left_on=query_id_col,
        right_index=True,
    )
    query_mod_df["relevant_docs_count"] = query_mod_df["relevant_docs_count"].fillna(0).astype(int)

    query_mod_df = query_mod_df.sort_values(
        by="relevant_docs_count",
        ascending=False,
    ).reset_index(drop=True)

    stats = {
        "Number of unique queries": f"{query_mod_df[query_id_col].nunique():,}",
        "Average query length (words)": f"{query_mod_df['query_length_words'].mean():.{digits}f}",
        "Average number of relevant documents per query": f"{query_mod_df['relevant_docs_count'].mean():.{digits}f}",
    }

    width = max(len(label) for label in stats)

    print("Basic query statistics")
    print("-" * (width + 20))
    for label, value in stats.items():
        print(f"{label:<{width}} : {value}")

    return stats, query_mod_df

In [31]:
test_query_df = pd.read_csv(queries_path("test"))

test_qrels = pd.read_csv(
    qrels_path("test"),
    sep="\t",
    header=None,
    names=["query_id", "unused", "doc_id", "relevance"],
)

_, query_mod_df = print_basic_query_stats(query_df=test_query_df, qrels=test_qrels)
display(query_mod_df.head(5), query_mod_df.sample(5), query_mod_df.tail(5))

Basic query statistics
------------------------------------------------------------------
Number of unique queries                       : 100
Average query length (words)                   : 2.07
Average number of relevant documents per query : 44.35


,id_left,text_left,query_length_words,relevant_docs_count
0,13540,united kingdom,2,2759
1,87860,brisbane,1,183
2,25174,berkshire,1,77
3,21145,miami,1,70
4,25726,northamptonshire,1,69


,id_left,text_left,query_length_words,relevant_docs_count
28,21645,north brabant,2,16
50,152444,manawatu wanganui,2,9
2,25174,berkshire,1,77
9,1368508,jutland,1,41
6,2161,cretaceous,1,55


,id_left,text_left,query_length_words,relevant_docs_count
95,34939,isleworth,1,6
96,410859,cerrado,1,6
97,362197,texas motor speedway,3,6
98,79032,elijah wood,2,6
99,996687,rom cartridge,2,6


---
# Running queries

In [48]:
def make_query_versions(
    query_df: pd.DataFrame,
    id_col: str = "id_left",
    text_col: str = "text_left",
    batch_size: int = 256
) -> dict[str, pd.DataFrame]:
    required_cols = [id_col, text_col]
    missing_cols = [col for col in required_cols if col not in query_df.columns]

    assert not missing_cols, f"Missing columns: {missing_cols}"
    assert not query_df[required_cols].isna().any().any(), f"Missing values found in {required_cols}"
    assert not query_df[id_col].duplicated().any(), f"Duplicate IDs found in {id_col}"

    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    stemmer = PorterStemmer()

    original_df = query_df[[id_col, text_col]].copy()

    stemmed_df = query_df[[id_col]].copy()
    stemmed_df[text_col] = [
        " ".join(stemmer.stem(word) for word in text.split())
        for text in tqdm(
            query_df[text_col],
            total=len(query_df),
        )
    ]

    lemmatized_df = query_df[[id_col]].copy()
    lemmatized_df[text_col] = [
        " ".join(token.lemma_ for token in doc if not token.is_space)
        for doc in tqdm(
            nlp.pipe(query_df[text_col], batch_size=batch_size),
            total=len(query_df),
        )
    ]

    return {
        "original": original_df,
        "stemmed": stemmed_df,
        "lemmatized": lemmatized_df,
    }

In [ ]:
test_query_versions = make_query_versions(query_df=test_query_df)

# из предыдущей дз
documents_versions = {
    "original": pd.read_csv(docs_path),
    "lemmatized": pd.read_csv(custom_path_documents("lemmatized")),
    "stemmed": pd.read_csv(custom_path_documents("stemmed")),
}

100%|██████████| 100/100 [00:00<00:00, 4625.39it/s]


In [84]:
def run_model(
    documents_df,
    queries_df,
    variant,
    model_type,
    top_k=1000,
    model_kwargs=None,
):
    model_kwargs = model_kwargs or {}

    doc_ids = documents_df["id_right"].tolist()
    doc_texts = documents_df["text_right"].tolist()

    query_ids = queries_df["id_left"].tolist()
    query_texts = queries_df["text_left"].tolist()

    start = time.perf_counter()

    if model_type == "tfidf":
        model = TfidfVectorizer(
            tokenizer=str.split,
            **model_kwargs,
        )
        doc_repr = model.fit_transform(doc_texts)

    if model_type == "bm25":
        model = BM25Okapi(
            doc_texts,
            tokenizer=str.split,
            **model_kwargs,
        )

    build_time = time.perf_counter() - start

    run_name = f"{model_type}_{variant}"
    rows = []
    query_times = []

    for query_id, query_text in tqdm(
        zip(query_ids, query_texts),
        total=len(query_ids),
        desc=run_name,
    ):
        start = time.perf_counter()

        if model_type == "tfidf":
            query_vec = model.transform([query_text])
            scores = (query_vec @ doc_repr.T).toarray()[0]
        if model_type == "bm25":
            scores = model.get_scores(query_text.split())

        query_times.append(time.perf_counter() - start)

        top_idx = np.argpartition(-scores, top_k - 1)[:top_k]
        ranked_idx = top_idx[np.argsort(-scores[top_idx])]

        for rank, idx in enumerate(ranked_idx, start=1):
            rows.append([
                query_id,
                "Q0",
                doc_ids[idx],
                rank,
                float(scores[idx]),
                run_name,
            ])

    run_df = pd.DataFrame(
        rows,
        columns=["query_id", "Q0", "doc_id", "rank", "score", "run_name"],
    )

    timing = {
        "model": model_type,
        "variant": variant,
        "build_time_sec": build_time,
        "total_query_time_sec": sum(query_times),
        "mean_query_time_ms": np.mean(query_times) * 1000,
    }

    return run_df, timing

def run_all_models(documents_versions, query_versions, runs_dir, top_k=1000):
    runs_dir.mkdir(parents=True, exist_ok=True)

    default_model_kwargs = {
        # l2-нормировка позволяет считать score через mat-vec произведение
        "tfidf": {
            "preprocessor": None,
            "token_pattern": None,
            "lowercase": False,
            "norm": "l2",
        },
        # базовые параметры
        "bm25": {
            "k1": 1.5,
            "b": 0.75,
            "epsilon": 0.25,
        },
    }

    all_runs = {}
    timing_rows = []

    for variant in ["original", "stemmed", "lemmatized"]:
        for model_type in ["tfidf", "bm25"]:
            run_df, timing = run_model(
                documents_df=documents_versions[variant],
                queries_df=query_versions[variant],
                variant=variant,
                model_type=model_type,
                top_k=top_k,
                model_kwargs=default_model_kwargs[model_type],
            )

            run_df.to_csv(
                runs_dir / f"{model_type}_{variant}.trec",
                sep=" ",
                index=False,
                header=False,
            )

            all_runs[f"{model_type}_{variant}"] = run_df
            timing_rows.append(timing)

    timing_df = pd.DataFrame(timing_rows)
    return all_runs, timing_df


In [85]:
all_runs, timing_df = run_all_models(
    documents_versions=documents_versions,
    query_versions=test_query_versions,
    runs_dir=runs_dir,
    top_k=1000,
)

display(timing_df)

bm25_lemmatized: 100%|██████████| 100/100 [00:13<00:00,  7.45it/s]


,model,variant,build_time_sec,total_query_time_sec,mean_query_time_ms
0,tfidf,original,11.950585,16.553518,165.535181
1,bm25,original,14.760758,12.911674,129.116738
2,tfidf,stemmed,10.468659,16.201770,162.017698
3,bm25,stemmed,17.001212,12.786357,127.863566
4,tfidf,lemmatized,10.368522,15.123435,151.234348
5,bm25,lemmatized,14.722727,13.260163,132.601630


---
# Evaluating runs against qrels

In [87]:
def evaluate_runs(all_runs, qrels_df):
    qrels_for_eval = qrels_df[["query_id", "doc_id", "relevance"]].copy()
    qrels_for_eval["query_id"] = qrels_for_eval["query_id"].astype(str)
    qrels_for_eval["doc_id"] = qrels_for_eval["doc_id"].astype(str)
    qrels_for_eval["relevance"] = qrels_for_eval["relevance"].astype(int)

    measures = {
        "P@1": P(rel=1, judged_only=False) @ 1,
        "P@10": P(rel=1, judged_only=False) @ 10,
        "P@20": P(rel=1, judged_only=False) @ 20,
        "MAP": AP(rel=1, judged_only=False),
        "nDCG@20": nDCG(
            dcg="log2",
            gains={0: 0, 1: 1, 2: 2},
            judged_only=False,
        ) @ 20,
    }

    evaluator = ir_measures.evaluator(list(measures.values()), qrels_for_eval)

    rows = []
    for run_name, run_df in all_runs.items():
        run_for_eval = run_df[["query_id", "doc_id", "score"]].copy()
        run_for_eval["query_id"] = run_for_eval["query_id"].astype(str)
        run_for_eval["doc_id"] = run_for_eval["doc_id"].astype(str)
        
        scores = evaluator.calc_aggregate(run_for_eval)

        row = {"run_name": run_name}
        for short_name, measure in measures.items():
            row[short_name] = scores[measure]
        rows.append(row)

    results_df = (
        pd.DataFrame(rows)
        .sort_values("run_name")
        .reset_index(drop=True)
        .round(4)
    )

    return results_df

In [88]:
results_df = evaluate_runs(all_runs=all_runs, qrels_df=test_qrels)
display(results_df)

,run_name,P@1,P@10,P@20,MAP,nDCG@20
0,bm25_lemmatized,0.48,0.210,0.1480,0.1755,0.3561
1,bm25_original,0.49,0.212,0.1500,0.1752,0.3570
2,bm25_stemmed,0.49,0.209,0.1455,0.1741,0.3540
3,tfidf_lemmatized,0.48,0.200,0.1300,0.1614,0.3391
4,tfidf_original,0.51,0.203,0.1350,0.1655,0.3499
5,tfidf_stemmed,0.50,0.194,0.1285,0.1595,0.3354


---
# Optimizing BM25

In [91]:
validation_query_df = pd.read_csv(queries_path("validation"))

validation_qrels = pd.read_csv(
    qrels_path("validation"),
    sep="\t",
    header=None,
    names=["query_id", "unused", "doc_id", "relevance"],
)

validation_query_versions = make_query_versions(query_df=validation_query_df)
_ = print_basic_query_stats(query_df=validation_query_df, qrels=validation_qrels)

100%|██████████| 100/100 [00:00<00:00, 4908.95it/s]

Basic query statistics
------------------------------------------------------------------
Number of unique queries                       : 100
Average query length (words)                   : 2.26
Average number of relevant documents per query : 49.79


In [92]:
def grid_search_bm25(
    documents_df,
    queries_df,
    qrels_df,
    metric,
    param_grid,
    variant="lemmatized",
    top_k=10,
):
    param_names = list(param_grid.keys())
    row_param, col_param = param_names

    qrels_for_eval = qrels_df[["query_id", "doc_id", "relevance"]].copy()
    qrels_for_eval["query_id"] = qrels_for_eval["query_id"].astype(str)
    qrels_for_eval["doc_id"] = qrels_for_eval["doc_id"].astype(str)
    qrels_for_eval["relevance"] = qrels_for_eval["relevance"].astype(int)

    evaluator = ir_measures.evaluator([metric], qrels_for_eval)

    rows = []

    for values in product(*(param_grid[param] for param in param_names)):
        model_kwargs = dict(zip(param_names, values))

        run_df, _ = run_model(
            documents_df=documents_df,
            queries_df=queries_df,
            variant=variant,
            model_type="bm25",
            top_k=top_k,
            model_kwargs=model_kwargs,
        )

        run_for_eval = run_df[["query_id", "doc_id", "score"]].copy()
        run_for_eval["query_id"] = run_for_eval["query_id"].astype(str)
        run_for_eval["doc_id"] = run_for_eval["doc_id"].astype(str)

        score = evaluator.calc_aggregate(run_for_eval)[metric]

        rows.append({
            row_param: model_kwargs[row_param],
            col_param: model_kwargs[col_param],
            str(metric): score,
        })

    results_df = pd.DataFrame(rows)

    grid_df = results_df.pivot(
        index=row_param,
        columns=col_param,
        values=str(metric),
    ).round(4)

    return grid_df


In [93]:
bm25_grid_df = grid_search_bm25(
    documents_df=documents_versions["lemmatized"],
    queries_df=validation_query_versions["lemmatized"],
    qrels_df=validation_qrels,
    metric=P(rel=1, judged_only=False) @ 10,
    param_grid={
        "k1": [0.6, 0.9, 1.2, 1.5, 1.8, 2.1],
        "b": [0.0, 0.25, 0.5, 0.75, 1.0],
    },
    variant="lemmatized",
    top_k=10,
)

display(bm25_grid_df)

bm25_lemmatized: 100%|██████████| 100/100 [00:14<00:00,  6.67it/s]


b,0.00,0.25,0.50,0.75,1.00
k1,,,,,
0.6,0.195,0.189,0.189,0.192,0.192
0.9,0.194,0.190,0.190,0.194,0.192
1.2,0.191,0.186,0.186,0.190,0.191
1.5,0.195,0.189,0.189,0.193,0.193
1.8,0.190,0.187,0.184,0.190,0.189
2.1,0.189,0.184,0.185,0.187,0.188


In [105]:
bm25_grid_with_mean = bm25_grid_df.copy()
bm25_grid_with_mean["mean"] = bm25_grid_with_mean.mean(axis=1)
bm25_grid_with_mean.loc["mean"] = bm25_grid_with_mean.mean(axis=0)

n_rows, n_cols = bm25_grid_df.shape

display(
    bm25_grid_with_mean.style
    .format("{:.4f}")
    .background_gradient(
        cmap="YlOrRd",
        axis=None,
        subset=pd.IndexSlice[bm25_grid_with_mean.index[:n_rows], bm25_grid_with_mean.columns[:n_cols]],
    )
)

b,0.000000,0.250000,0.500000,0.750000,1.000000,mean
k1,,,,,,
0.600000,0.1950,0.1890,0.1890,0.1920,0.1920,0.1914
0.900000,0.1940,0.1900,0.1900,0.1940,0.1920,0.1920
1.200000,0.1910,0.1860,0.1860,0.1900,0.1910,0.1888
1.500000,0.1950,0.1890,0.1890,0.1930,0.1930,0.1918
1.800000,0.1900,0.1870,0.1840,0.1900,0.1890,0.1880
2.100000,0.1890,0.1840,0.1850,0.1870,0.1880,0.1866
mean,0.1923,0.1875,0.1872,0.1910,0.1908,0.1898


In [107]:
top_2_params = (
    bm25_grid_df.stack()
    .sort_values(ascending=False)
    .head(2)
    .reset_index()
)
top_2_params.columns = ["k1", "b", "validation_P@10"]

all_runs_tuned = {}
timing_rows = []

for _, row in top_2_params.iterrows():
    k1 = row["k1"]
    b = row["b"]
    run_name = f"bm25_lemmatized_tuned_k1_{k1}_b_{b}"

    tuned_run, tuned_timing = run_model(
        documents_df=documents_versions["lemmatized"],
        queries_df=test_query_versions["lemmatized"],
        variant="lemmatized",
        model_type="bm25",
        top_k=1000,
        model_kwargs={
            "k1": k1,
            "b": b,
        },
    )

    all_runs_tuned[run_name] = tuned_run
    timing_rows.append(tuned_timing)

tuned_results_df = evaluate_runs(
    all_runs=all_runs_tuned,
    qrels_df=test_qrels,    
)

display(top_2_params)
display(tuned_results_df)


bm25_lemmatized: 100%|██████████| 100/100 [00:16<00:00,  6.07it/s]


,k1,b,validation_P@10
0,0.6,0.0,0.195
1,1.5,0.0,0.195


,run_name,P@1,P@10,P@20,MAP,nDCG@20
0,bm25_lemmatized_tuned_k1_0.6_b_0.0,0.48,0.207,0.1430,0.1716,0.3474
1,bm25_lemmatized_tuned_k1_1.5_b_0.0,0.48,0.212,0.1465,0.1759,0.3552


---
# Some analysis